# 1 load

In [ ]:
#imports
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#paths
#DATA_DIR = Path("../data")   #αν το notebook είναι μέσα σε notebooks/
DATA_DIR = Path((__import__('os').environ.get('SHROOM_PROJECT_ROOT', '.') + '/data'))    #αν το notebook είναι στο root

#unlabeled train
train_agnostic_path = DATA_DIR / "Unlabeled train" / "SHROOM_unlabeled-training-data-v2" / "train.model-agnostic.json"
train_aware_path = DATA_DIR / "Unlabeled train" / "SHROOM_unlabeled-training-data-v2" / "train.model-aware.v2.json"
#trial
trial_path = DATA_DIR / "Trial" / "SHROOM_trial-v1.1" / "trial-v1.json"
#labeled test
test_agnostic_path = DATA_DIR / "Labeled test" / "SHROOM_test-labeled" / "test.model-agnostic.json"
test_aware_path = DATA_DIR / "Labeled test" / "SHROOM_test-labeled" / "test.model-aware.json"

In [ ]:
#check paths
print(train_agnostic_path)
print(train_aware_path)
print(trial_path)
print(test_agnostic_path)
print(test_aware_path)

In [ ]:
#function to load json data into a pandas dataframe
def load_json_to_df(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    return df

#data frames
train_agnostic_df = load_json_to_df(train_agnostic_path)
train_aware_df = load_json_to_df(train_aware_path)
trial_df = load_json_to_df(trial_path)
test_agnostic_df = load_json_to_df(test_agnostic_path)
test_aware_df = load_json_to_df(test_aware_path)

#print shapes of dataframes
print("Train agnostic shape:", train_agnostic_df.shape)
print("Train aware shape:", train_aware_df.shape)
print("Trial shape:", trial_df.shape)
print("Test agnostic shape:", test_agnostic_df.shape)
print("Test aware shape:", test_aware_df.shape)

#print heads of dataframes
#display(train_agnostic_df.head(2))
#display(train_aware_df.head(2))
#display(trial_df.head(2))
#display(test_agnostic_df.head(2))
#display(test_aware_df.head(2))

# 2 EDA

In [ ]:
#label distribution
def label_distribution(df, name):
    print("\n----------------------------------")
    print(f"{name} - Label Distribution:")
    print("--Shape:", df.shape)
    print("--Counts:")
    print(df["label"].value_counts(dropna=False).to_string())
    print("--Percentages:")
    print((df["label"].value_counts(normalize=True, dropna=False) * 100).round(2).to_string())
    print("----------------------------------")

label_distribution(trial_df, "Trial")
label_distribution(test_agnostic_df, "Test Agnostic")
label_distribution(test_aware_df, "Test Aware")


In [ ]:
#label by task distribution
def label_distribution_by_task(df, name):
    print("\n--------------------------------------------")
    print(f"{name} - Labels by Task Distribution:")
    print("--Shape:", df.shape)

    counts = pd.crosstab(df["task"], df["label"])
    perc = pd.crosstab(df["task"], df["label"], normalize="index") * 100

    print("--Counts:")
    print(counts.to_string())

    print("--Percentages:")
    print(perc.round(2).to_string())

    print("--------------------------------------------")

label_distribution_by_task(trial_df, "Trial")
label_distribution_by_task(test_agnostic_df, "Test Agnostic")
label_distribution_by_task(test_aware_df, "Test Aware")

In [ ]:
# binned difficulty distribution based on p(Hallucination)
def difficulty_distribution(df, name, prob_col="p(Hallucination)"):
    print("\n--------------------------------------------")
    print(f"{name} - Difficulty Distribution:")
    print("--Shape:", df.shape)

    bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    labels = ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]

    difficulty = pd.cut(
        df[prob_col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    counts = difficulty.value_counts(sort=False)
    perc = (difficulty.value_counts(sort=False, normalize=True) * 100).round(2)

    print("--Counts:")
    print(counts.to_string())

    print("--Percentages:")
    print(perc.to_string())

    print("--------------------------------------------")

# difficulty distribution by task
def difficulty_distribution_by_task(df, name, prob_col="p(Hallucination)"):
    print("\n------------------------------------------------------")
    print(f"{name} - Difficulty by Task Distribution:")
    print("--Shape:", df.shape)

    bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    labels = ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]

    df_temp = df.copy()
    df_temp["difficulty_bin"] = pd.cut(
        df_temp[prob_col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    counts = pd.crosstab(df_temp["task"], df_temp["difficulty_bin"])
    perc = pd.crosstab(df_temp["task"], df_temp["difficulty_bin"], normalize="index") * 100

    print("--Counts:")
    print(counts.to_string())

    print("--Percentages:")
    print(perc.round(2).to_string())

    print("------------------------------------------------------")

difficulty_distribution(trial_df, "Trial")
difficulty_distribution(test_agnostic_df, "Test Agnostic")
difficulty_distribution(test_aware_df, "Test Aware")

difficulty_distribution_by_task(trial_df, "Trial")
difficulty_distribution_by_task(test_agnostic_df, "Test Agnostic")
difficulty_distribution_by_task(test_aware_df, "Test Aware")

In [ ]:
!pip install tiktoken protobuf sentencepiece transformers

#import sys
#print(sys.executable)
#!{sys.executable} -m pip install -U sentencepiece protobuf transformers

#import google.protobuf
#import sentencepiece
#import transformers
#print("protobuf:", google.protobuf.__version__)
#print("sentencepiece:", sentencepiece.__version__)
#print("transformers:", transformers.__version__)

In [ ]:
#tokenized distribution
#class AutoTokenizer:
from transformers import AutoTokenizer
#object tokenizer,  
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

MAX_LEN = 512

In [ ]:
##single text token length
def token_len(text):
    if pd.isna(text):
        return 0
    return len(tokenizer.encode(str(text), add_special_tokens=False))

#pair text token length
def pair_token_len(text1, text2):
    text1 = "" if pd.isna(text1) else str(text1)
    text2 = "" if pd.isna(text2) else str(text2)

    encoded = tokenizer(
        text1,
        text2,
        truncation=False,
        add_special_tokens=True
    )
    return len(encoded["input_ids"])

In [ ]:
#add token length columns
def add_token_length_columns(df):
    df = df.copy()

    if "src" in df.columns:
        df["src_tokens"] = df["src"].apply(token_len)

    if "hyp" in df.columns:
        df["hyp_tokens"] = df["hyp"].apply(token_len)

    if "tgt" in df.columns:
        df["tgt_tokens"] = df["tgt"].apply(token_len)

    if "src" in df.columns and "hyp" in df.columns:
        df["src_hyp_tokens"] = df.apply(lambda x: pair_token_len(x["src"], x["hyp"]), axis=1)

    if "tgt" in df.columns and "hyp" in df.columns:
        df["tgt_hyp_tokens"] = df.apply(lambda x: pair_token_len(x["tgt"], x["hyp"]), axis=1)

    return df

In [ ]:
#apply token length columns
trial_df = add_token_length_columns(trial_df)
test_agnostic_df = add_token_length_columns(test_agnostic_df)
test_aware_df = add_token_length_columns(test_aware_df)

In [ ]:
#token length distribution
def token_length_distribution(df, name):
    print("\n----------------------------------")
    print(f"{name} - Token Length Distribution:")
    print("--Shape:", df.shape)

    cols = ["src_tokens", "hyp_tokens", "tgt_tokens", "src_hyp_tokens", "tgt_hyp_tokens"]

    for col in cols:
        if col in df.columns:
            print(f"\n--{col}:")
            print(df[col].describe(percentiles=[0.50, 0.90, 0.95, 0.99]).round(2).to_string())

    print("----------------------------------")

In [ ]:
#run token length by distribution
token_length_distribution(trial_df, "Trial")
token_length_distribution(test_agnostic_df, "Test Agnostic")
token_length_distribution(test_aware_df, "Test Aware")

# 3 NLI BASELINE

In [ ]:
!pip install transformers torch scikit-learn pandas numpy scipy tqdm
%pip install torch transformers

In [ ]:
#imports
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from transformers import pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr

In [ ]:
#checking which dataframe has label or labels column
dfs = {
    "train_agnostic_df": train_agnostic_df,
    "train_aware_df": train_aware_df,
    "trial_df": trial_df,
    "test_agnostic_df": test_agnostic_df,
    "test_aware_df": test_aware_df
}

for name, temp_df in dfs.items():
    print("\n" + name)
    print("shape:", temp_df.shape)
    print("columns:", temp_df.columns.tolist())

In [ ]:
#Choosing dataset
#Για το πρώτο NLI baseline δουλεύουμε στο agnostic labeled set
df = test_agnostic_df.copy()

label_col = "label"
prob_col = "p(Hallucination)"

print(df.shape)
print(df.columns.tolist())

df[["task", "src", "hyp", "tgt", label_col, prob_col]].head()

In [ ]:
#Targets
#Making the targets for training and evaluation

#Target for regression
df["p_hallucination"] = df[prob_col]

#Target for binary accuracy
df["label_bin"] = df[label_col].map({
    "Hallucination": 1,
    "Not Hallucination": 0
})

#head
print(df[ [label_col, "label_bin", "p_hallucination"] ].head())
print()
print(df["p_hallucination"].value_counts(dropna=False))
print()
print(df["label_bin"].value_counts(dropna=False))

In [ ]:
#Load NLI model
#NLI model will give us probabilities for entailment, neutral, contradiction which we can use as features in our ensemble 
device = -1  # -1 for CPU και 0 για GPU (αν υπάρχει διαθέσιμη)

nli_model_name = "microsoft/deberta-large-mnli"

nli_pipe = pipeline(
    "text-classification",
    model=nli_model_name,
    tokenizer=nli_model_name,
    truncation=True,
    max_length=512,
    device=device,
    top_k=None   #επιστρέφει όλα τα labels, όχι μόνο το καλύτερο
)

In [ ]:
#Quick NLI check
test_result = nli_pipe({
    "text": "A man is walking in the park.",
    "text_pair": "A person is walking outside."
})

test_result

In [ ]:
#Function to get NLI scores
#returns clear the three NLI scores

#Συνάρτηση που παίρνει δύο προτάσεις:
# premise: η αρχική / βασική πρόταση
# hypothesis: η πρόταση που θέλουμε να συγκρίνουμε με την αρχική
# pipe: το έτοιμο NLI μοντέλο / pipeline, φορτώθηκε πριν δύο cells
#
#Η συνάρτηση επιστρέφει τρία scores:
# entailment, neutral, contradiction

def get_nli_scores(premise, hypothesis, pipe):

    #Στέλνουμε τις δύο προτάσεις στο NLI μοντέλο.
    #Το "text" είναι το premise.
    #Το "text_pair" είναι το hypothesis.
    #Χρησιμοποιούμε str() για να είμαστε σίγουροι ότι είναι κείμενο.
    result = pipe({
        "text": str(premise),
        "text_pair": str(hypothesis)
    })

    #Κάποιες εκδόσεις του transformers επιστρέφουν το αποτέλεσμα
    #μέσα σε διπλή λίστα, δηλαδή:
    # [[ {...}, {...}, {...} ]]
    #
    #Αν συμβεί αυτό, κρατάμε μόνο την εσωτερική λίστα:
    # [ {...}, {...}, {...} ]
    if isinstance(result, list) and len(result) > 0 and isinstance(result[0], list):
        result = result[0]

    #Μετατρέπουμε τη λίστα αποτελεσμάτων σε dictionary.
    #Παράδειγμα:
    #Από:
    # [
    #   {"label": "entailment", "score": 0.80},
    #   {"label": "neutral", "score": 0.15},
    #   {"label": "contradiction", "score": 0.05}
    # ]
    #
    #Γίνεται:
    # {
    #   "entailment": 0.80,
    #   "neutral": 0.15,
    #   "contradiction": 0.05
    # }

    #Το lower() κάνει τα labels πεζά, για να μη μας νοιάζει
    #αν το μοντέλο επιστρέψει "ENTAILMENT" ή "entailment".
    scores = {
        item["label"].lower(): item["score"]
        for item in result
    }

    #Παίρνουμε ξεχωριστά το score για entailment.
    #Αν δεν υπάρχει τέτοιο label, επιστρέφουμε 0.0.
    ent = scores.get("entailment", 0.0)

    #Παίρνουμε ξεχωριστά το score για neutral.
    #Αν δεν υπάρχει τέτοιο label, επιστρέφουμε 0.0.
    neu = scores.get("neutral", 0.0)

    #Παίρνουμε ξεχωριστά το score για contradiction.
    #Αν δεν υπάρχει τέτοιο label, επιστρέφουμε 0.0.
    con = scores.get("contradiction", 0.0)

    #Επιστρέφουμε τα τρία scores με σταθερή σειρά:
    #1. entailment
    #2. neutral
    #3. contradiction
    return ent, neu, con

In [ ]:
#Check score function
ent, neu, con = get_nli_scores(
    "A man is walking in the park.",
    "A person is walking outside.",
    nli_pipe
)

print("Entailment:", ent)
print("Neutral:", neu)
print("Contradiction:", con) 

In [ ]:
#Time estimation
import time

sample_df = df.head(20).copy()

start_time = time.time()

#Για κάθε γραμμή του sample_df, πάρε τη γραμμή ως row, αγνόησε το index (_), και δείξε μπάρα προόδου μέχρι να τελειώσουν όλες οι γραμμές.
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    #function calls
    get_nli_scores(row["src"], row["hyp"], nli_pipe)
    get_nli_scores(row["tgt"], row["hyp"], nli_pipe)

end_time = time.time()

elapsed_time = end_time - start_time
time_per_sample = elapsed_time / len(sample_df)
estimated_total_time = time_per_sample * len(df)

print(f"Χρόνος για 20 δείγματα: {elapsed_time:.2f} δευτερόλεπτα")
print(f"Μέσος χρόνος ανά δείγμα: {time_per_sample:.2f} δευτερόλεπτα")
print(f"Εκτίμηση για όλο το dataset: {estimated_total_time / 60:.2f} λεπτά")

In [ ]:
#Full NLI feature extraction
#Extracting the 6 NLI features for each sample: src_ent, src_neu, src_con, tgt_ent, tgt_neu, tgt_con

#Λίστες όπου θα αποθηκεύσουμε τα NLI scores για κάθε δείγμα
src_ent_list = []
src_neu_list = []
src_con_list = []

tgt_ent_list = []
tgt_neu_list = []
tgt_con_list = []

#Τρέχουμε NLI σε όλο το dataframe
for _, row in tqdm(df.iterrows(), total=len(df)):

    # Pair 1: source -> hypothesis
    src_ent, src_neu, src_con = get_nli_scores(
        row["src"],
        row["hyp"],
        nli_pipe
    )

    # Pair 2: target -> hypothesis
    tgt_ent, tgt_neu, tgt_con = get_nli_scores(
        row["tgt"],
        row["hyp"],
        nli_pipe
    )

    src_ent_list.append(src_ent)
    src_neu_list.append(src_neu)
    src_con_list.append(src_con)

    tgt_ent_list.append(tgt_ent)
    tgt_neu_list.append(tgt_neu)
    tgt_con_list.append(tgt_con)

#Προσθέτουμε τα 6 NLI features στο dataframe
df["src_ent"] = src_ent_list
df["src_neu"] = src_neu_list
df["src_con"] = src_con_list

df["tgt_ent"] = tgt_ent_list
df["tgt_neu"] = tgt_neu_list
df["tgt_con"] = tgt_con_list

#Έλεγχος ότι δημιουργήθηκαν σωστά
df[[
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con"
]].head()

In [ ]:
#Saving the dataset with NLI features
df.to_csv("nli_features_dataset.csv", index=False)

# Random Forest baseline

In [ ]:
#Φτιάχνουμε το feature matrix και τα targets για το Random Forest baseline.

nli_feature_cols  = [
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con"
]

#Φτιάχνει το feature matrix με τις 6 NLI στήλες
nli_X = df[nli_feature_cols ].values
#Eίναι το regression target. Δηλαδή κρατάει πιθανότητα hallucination: 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
y_reg = df["p_hallucination"].values
#Eίναι το classification target. Δηλαδή 1 για Hallucination και 0 για Not Hallucination
y_bin = df["label_bin"].values
#Kρατάει το task κάθε δείγματος. Δηλαδή MT, DM, PG
tasks = df["task"].values

print("nli_X shape:", nli_X.shape)
print("y_reg shape:", y_reg.shape)
print("y_bin shape:", y_bin.shape)

In [ ]:
#Eκπαιδεύουμε Random Forest με 5-fold cross-validation πάνω στα NLI features.

#Χωρίζει το dataset σε 5 folds, κρατώντας περίπου ίδια αναλογία Hallucination / Not Hallucination σε κάθε fold
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

all_val_preds = []
all_val_true_reg = []
all_val_true_bin = []
all_val_tasks = []

#Δημιουργεί ένα σύνθετο stratification key που συνδυάζει task και label.
#Έτσι κάθε fold κρατά περίπου την ίδια κατανομή όχι μόνο σε Hallucination / Not Hallucination,
#αλλά και ανά task (DM, MT, PG), κάτι σημαντικό γιατί τα tasks έχουν διαφορετική δυσκολία.
stratify_key = df["task"].astype(str) + "_" + df["label_bin"].astype(str)
#Για κάθε fold του cross-validation, πάρε τα training indexes και τα validation indexes και τρέξε το μοντέλο.
for fold, (train_idx, val_idx) in enumerate(skf.split(nli_X, stratify_key), start=1):
    X_train = nli_X[train_idx]
    X_val = nli_X[val_idx]

    #Χωρίζει τα features σε training και validation set
    y_train_reg = y_reg[train_idx]
    y_val_reg = y_reg[val_idx]
    #Χωρίζει το regression target, δηλαδή το p_hallucination
    y_val_bin = y_bin[val_idx]

    #Κρατάει τα πραγματικά binary labels για το validation set, για να υπολογίσουμε accuracy
    val_tasks = tasks[val_idx]

    #Εκπαιδεύει Random Forest regressor, όχι classifier. Δηλαδή μαθαίνει να προβλέπει: p_hallucination: 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
    model = RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train_reg)

    #Mετατρέπει την προβλεπόμενη πιθανότητα σε binary label
    #>= 0.5  -> Hallucination
    #< 0.5   -> Not Hallucination
    val_pred = model.predict(X_val)
    val_pred = np.clip(val_pred, 0.0, 1.0)

    # Για accuracy μετατρέπουμε την πιθανότητα σε binary label.
    val_pred_bin = (val_pred >= 0.5).astype(int)

    #Yπολογίζει classification accuracy
    acc = accuracy_score(y_val_bin, val_pred_bin)
    #Mετράει πόσο καλά η προβλεπόμενη πιθανότητα ακολουθεί τη σειρά/βαθμό του πραγματικού p_hallucination
    sp = spearmanr(y_val_reg, val_pred).correlation

    fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "spearman": sp
    })

    all_val_preds.extend(val_pred)
    all_val_true_reg.extend(y_val_reg)
    all_val_true_bin.extend(y_val_bin)
    all_val_tasks.extend(val_tasks)

nli_results_df = pd.DataFrame(fold_results)
nli_results_df

In [ ]:
#Yπολογίζουμε συνολικά Accuracy και Spearman για τα out-of-fold predictions.

overall_pred = np.array(all_val_preds)
overall_true_reg = np.array(all_val_true_reg)
overall_true_bin = np.array(all_val_true_bin)

overall_pred_bin = (overall_pred >= 0.5).astype(int)

overall_acc = accuracy_score(overall_true_bin, overall_pred_bin)
overall_sp = spearmanr(overall_true_reg, overall_pred).correlation

print("Mean CV Accuracy:", nli_results_df["accuracy"].mean())
print("Mean CV Spearman:", nli_results_df["spearman"].mean())
print("Overall OOF Accuracy:", overall_acc)
print("Overall OOF Spearman:", overall_sp)

In [ ]:
#Eλέγχουμε την απόδοση ξεχωριστά για κάθε task.

nli_oof_df = pd.DataFrame({
    "task": all_val_tasks,
    "y_true_reg": all_val_true_reg,
    "y_true_bin": all_val_true_bin,
    "y_pred_reg": all_val_preds
})

nli_oof_df["y_pred_bin"] = (nli_oof_df["y_pred_reg"] >= 0.5).astype(int)

for task_name in sorted(nli_oof_df["task"].unique()):

    task_df = nli_oof_df[nli_oof_df["task"] == task_name]

    acc = accuracy_score(
        task_df["y_true_bin"],
        task_df["y_pred_bin"]
    )

    sp = spearmanr(
        task_df["y_true_reg"],
        task_df["y_pred_reg"]
    ).correlation

    print(f"{task_name} -> Accuracy: {acc:.4f}, Spearman: {sp:.4f}")

In [ ]:
#Αποθηκεύουμε predictions και αποτελέσματα για μελλοντικό error analysis.

nli_oof_df.to_csv("nli_baseline_oof_predictions.csv", index=False)
nli_results_df.to_csv("nli_baseline_cv_results.csv", index=False)

## NLI Baseline - Results

Το πρώτο baseline χρησιμοποιεί μόνο τα 6 NLI features:
- src_ent, src_neu, src_con
- tgt_ent, tgt_neu, tgt_con

Τα καλύτερα αποτελέσματα εμφανίζονται στο MT, όπου το NLI μπορεί να εντοπίσει πιο καθαρές σημασιολογικές αποκλίσεις.  
Το DM έχει χαμηλότερο Accuracy, κάτι που δείχνει ότι πιθανόν χρειάζεται εξωτερική γνώση ή επιπλέον features.
Το PG έχει καλό Accuracy αλλά χαμηλότερο Spearman, άρα το μοντέλο ταξινομεί σωστά αρκετά δείγματα αλλά δεν αποτυπώνει τέλεια τη βαθμίδα αβεβαιότητας.

# 4 SEMANTIC SIMILARITY FEATURES


In [ ]:
#sentence embeddings library
%pip install sentence-transformers

In [ ]:
#Φορτώνουμε το μοντέλο που θα υπολογίσει τα sentence embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

sim_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [ ]:
#cleaning text fields
text_cols = ["src", "hyp", "tgt"]

for col in text_cols:
    df[col] = df[col].fillna("").astype(str)

In [ ]:
#calculating embeddings για src, hyp, tgt
src_emb = sim_model.encode(
    df["src"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

hyp_emb = sim_model.encode(
    df["hyp"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

tgt_emb = sim_model.encode(
    df["tgt"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

In [ ]:
#Calculating the 3 basic similarity features: sim_src_hyp, sim_tgt_hyp, sim_src_tgtd
df["sim_src_hyp"] = np.sum(src_emb * hyp_emb, axis=1)
df["sim_tgt_hyp"] = np.sum(tgt_emb * hyp_emb, axis=1)
df["sim_src_tgt"] = np.sum(src_emb * tgt_emb, axis=1)

df[["sim_src_hyp", "sim_tgt_hyp", "sim_src_tgt"]].head()

In [ ]:
#Adding some similarity features
df["sim_max"] = df[["sim_src_hyp", "sim_tgt_hyp"]].max(axis=1)
df["sim_min"] = df[["sim_src_hyp", "sim_tgt_hyp"]].min(axis=1)
df["sim_mean"] = df[["sim_src_hyp", "sim_tgt_hyp"]].mean(axis=1)

df["sim_gap_tgt_src"] = df["sim_tgt_hyp"] - df["sim_src_hyp"]
df["sim_hyp_distance_from_truth"] = 1.0 - df["sim_tgt_hyp"]

In [ ]:
#Saving the dataset with NLI + similarity
df.to_csv("nli_similarity_features_dataset.csv", index=False)

In [ ]:
#Define feature groups for the NLI + Semantic Similarity experiment

nli_cols = [
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con"
]

sim_cols = [
    "sim_src_hyp",
    "sim_tgt_hyp",
    "sim_src_tgt",
    "sim_max",
    "sim_min",
    "sim_mean",
    "sim_gap_tgt_src",
    "sim_hyp_distance_from_truth"
]

nli_sim_feature_cols  = nli_cols + sim_cols

nli_sim_X  = df[nli_sim_feature_cols ].values
y_reg = df["p_hallucination"].values
y_bin = df["label_bin"].values
tasks = df["task"].values

print("nli_sim_X shape:", nli_sim_X.shape)

In [ ]:
#Ελέγχουμε ότι το dataframe περιέχει ήδη τα NLI features.

print(df.shape)

df[[
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con"
]].head()

In [ ]:
#Εκπαιδεύουμε Random Forest με 5-fold cross-validation πάνω στα NLI + Similarity features
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

all_val_preds = []
all_val_true_reg = []
all_val_true_bin = []
all_val_tasks = []

stratify_key = df["task"].astype(str) + "_" + df["label_bin"].astype(str)

for fold, (train_idx, val_idx) in enumerate(skf.split(nli_sim_X, stratify_key), start=1):
    #Χωρίζουμε τα δεδομένα του συγκεκριμένου fold σε train και validation.
    X_train = nli_sim_X[train_idx]
    X_val = nli_sim_X[val_idx]

    y_train_reg = y_reg[train_idx]
    y_val_reg = y_reg[val_idx]

    y_val_bin = y_bin[val_idx]
    val_tasks = tasks[val_idx]

    #Δημιουργούμε το Random Forest μοντέλο.
    model = RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    #Εκπαιδεύουμε το μοντέλο στο training set.
    model.fit(X_train, y_train_reg)

    #Το μοντέλο προβλέπει πιθανότητα hallucination.
    val_pred = model.predict(X_val)

    #Περιορίζουμε τις προβλέψεις στο διάστημα [0, 1].
    val_pred = np.clip(val_pred, 0.0, 1.0)

    #Μετατρέπουμε την πιθανότητα σε binary label.
    val_pred_bin = (val_pred >= 0.5).astype(int)

    #Υπολογίζουμε accuracy και Spearman correlation.
    acc = accuracy_score(y_val_bin, val_pred_bin)
    sp = spearmanr(y_val_reg, val_pred).correlation

    #Αποθηκεύουμε τα αποτελέσματα του fold.
    fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "spearman": sp
    })

    #Αποθηκεύουμε τις προβλέψεις για συνολική αξιολόγηση.
    all_val_preds.extend(val_pred)
    all_val_true_reg.extend(y_val_reg)
    all_val_true_bin.extend(y_val_bin)
    all_val_tasks.extend(val_tasks)

#Μετατρέπουμε τα αποτελέσματα των folds σε DataFrame.
nli_sim_results_df  = pd.DataFrame(fold_results)

nli_sim_results_df

In [ ]:
#Per task analysis
nli_sim_oof_df  = pd.DataFrame({
    "task": all_val_tasks,
    "y_true_reg": all_val_true_reg,
    "y_true_bin": all_val_true_bin,
    "y_pred_reg": all_val_preds
})

nli_sim_oof_df["y_pred_bin"] = (nli_sim_oof_df["y_pred_reg"] >= 0.5).astype(int)

for task_name in sorted(nli_sim_oof_df["task"].unique()):

    task_df = nli_sim_oof_df[nli_sim_oof_df["task"] == task_name]

    acc = accuracy_score(
        task_df["y_true_bin"],
        task_df["y_pred_bin"]
    )

    sp = spearmanr(
        task_df["y_true_reg"],
        task_df["y_pred_reg"]
    ).correlation

    print(f"{task_name} -> Accuracy: {acc:.4f}, Spearman: {sp:.4f}")

In [ ]:
#Feature importance
final_model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

final_model.fit(nli_sim_X, y_reg)

importance_df = pd.DataFrame({
    "feature": nli_sim_feature_cols,
    "importance": final_model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df

In [ ]:
#Υπολογίζουμε τον μέσο όρο των αποτελεσμάτων από τα 5 folds.
print("Mean CV Accuracy:", nli_sim_results_df ["accuracy"].mean())
print("Mean CV Spearman:", nli_sim_results_df["spearman"].mean())

In [ ]:
#Υπολογίζουμε συνολικά αποτελέσματα πάνω σε όλες τις validation προβλέψεις.

all_val_preds = np.array(all_val_preds)
all_val_true_reg = np.array(all_val_true_reg)
all_val_true_bin = np.array(all_val_true_bin)
all_val_tasks = np.array(all_val_tasks)

all_val_pred_bin = (all_val_preds >= 0.5).astype(int)

overall_acc = accuracy_score(all_val_true_bin, all_val_pred_bin)
overall_sp = spearmanr(all_val_true_reg, all_val_preds).correlation

print("Overall OOF Accuracy:", overall_acc)
print("Overall OOF Spearman:", overall_sp)

In [ ]:
# Save NLI + Similarity baseline results
# Τρέξε αυτό το cell αφού έχουν δημιουργηθεί τα nli_sim_results_df και nli_sim_oof_df.

from pathlib import Path

OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

nli_sim_cv_path = OUTPUT_DIR / "nli_sim_baseline_cv_results.csv"
nli_sim_oof_path = OUTPUT_DIR / "nli_sim_baseline_oof_predictions.csv"

nli_sim_results_df.to_csv(nli_sim_cv_path, index=False)
nli_sim_oof_df.to_csv(nli_sim_oof_path, index=False)

print(f"Saved CV results to: {nli_sim_cv_path.resolve()}")
print(f"Saved OOF predictions to: {nli_sim_oof_path.resolve()}")

display(nli_sim_results_df)
display(nli_sim_oof_df.head())

In [ ]:
#Υπολογίζουμε τα αποτελέσματα ξεχωριστά για κάθε task.

for task in sorted(set(all_val_tasks)):

    task_mask = all_val_tasks == task

    task_acc = accuracy_score(
        all_val_true_bin[task_mask],
        all_val_pred_bin[task_mask]
    )

    task_sp = spearmanr(
        all_val_true_reg[task_mask],
        all_val_preds[task_mask]
    ).correlation

    print(task, "-> Accuracy:", round(task_acc, 4), "Spearman:", round(task_sp, 4))

# 5. Data Augmentation

Σε αυτό το στάδιο δημιουργούμε συνθετικά training samples με weak supervision. Για κάθε unlabeled sample ζητάμε από το Gemini δύο εξόδους: μία faithful έξοδο που πρέπει να είναι σωστή παράφραση του reference και μία hallucinated έξοδο που πρέπει να είναι ρεαλιστική αλλά μη υποστηριζόμενη. Τα synthetic labels δεν θεωρούνται απόλυτα αξιόπιστα, οπότε στη συνέχεια εφαρμόζεται strict filtering και δημιουργείται balanced synthetic dataset για χρήση στο training.


In [ ]:
#install Gemini SDK
!pip install -q -U google-genai

In [ ]:
#imports and augmentation settings
#Κρατάμε τα augmentation imports και το βασικό config συγκεντρωμένα εδώ.
#Για μικρό/μεγαλύτερο run αλλάζουμε κυρίως SAMPLES_PER_TASK και BATCH_SIZE.

import os
import re
import json
import time
import datetime
from pathlib import Path
from getpass import getpass

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from google import genai
from google.genai import types

#Αριθμός αρχικών samples που παίρνουμε από κάθε task (DM, MT, PG).
#Για δοκιμές κρατάμε μικρή τιμή. Για μεγαλύτερο run αυξάνεται σταδιακά.
SAMPLES_PER_TASK = 200

#Πόσα samples στέλνουμε σε κάθε Gemini request.
#Μικρό batch είναι πιο σταθερό, μεγάλο batch μειώνει τα API requests.
BATCH_SIZE = 2

#Μικρή αναμονή μετά από κάθε batch ώστε να μη χτυπάμε συνεχώς το API.
REQUEST_SLEEP = 3.0

#Fail-fast retries: δεν περιμένουμε πολλά λεπτά σε αποτυχημένο batch.
MAX_BATCH_RETRIES = 2
RETRY_SLEEP_SECONDS = 5

#Αν ένα batch αποτύχει πλήρως, το γράφουμε στο failed log και συνεχίζουμε.
SKIP_FAILED_BATCHES = True

#Gemini models.
#Το primary είναι το φθηνότερο/ελαφρύτερο μοντέλο για πολλά μικρά generations.
#Το fallback χρησιμοποιείται μόνο αν το primary έχει θέμα ποιότητας/διαθεσιμότητας.
AUG_MODEL_PRIMARY = "gemini-2.5-flash-lite"
AUG_MODEL_FALLBACK = "gemini-2.5-flash"

#Επιλογή μοντέλου για το τρέχον run.
USE_FALLBACK_MODEL = False
AUG_MODEL = AUG_MODEL_FALLBACK if USE_FALLBACK_MODEL else AUG_MODEL_PRIMARY

#Smoke test.
#Το κρατάμε False by default γιατί κάθε κλήση καταναλώνει quota.
RUN_SMOKE_TEST = False

#Output directory για checkpoints και synthetic data.
#Όλα τα intermediate/final αρχεία του augmentation γράφονται εδώ.
AUG_OUTPUT_DIR = Path("augmentation_outputs_vertex")
AUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n================================")
print("Augmentation Configuration:")
print(f"SAMPLES_PER_TASK: {SAMPLES_PER_TASK}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"REQUEST_SLEEP: {REQUEST_SLEEP}s")
print(f"MAX_BATCH_RETRIES: {MAX_BATCH_RETRIES}")
print(f"RETRY_SLEEP_SECONDS: {RETRY_SLEEP_SECONDS}s")
print(f"SKIP_FAILED_BATCHES: {SKIP_FAILED_BATCHES}")
print(f"AUG_MODEL_PRIMARY: {AUG_MODEL_PRIMARY}")
print(f"AUG_MODEL_FALLBACK: {AUG_MODEL_FALLBACK}")
print(f"USE_FALLBACK_MODEL: {USE_FALLBACK_MODEL}")
print(f"Selected Gemini model: {AUG_MODEL}")
print("Augmentation output dir:", AUG_OUTPUT_DIR.resolve())
print("================================\n")


In [ ]:
#Gemini / Vertex AI setup with service account

from pathlib import Path
from google import genai
from google.genai import types
from google.oauth2 import service_account

#Path στο JSON key.
#Μην ανεβάσεις αυτό το αρχείο σε GitHub ή ChatGPT.
SERVICE_ACCOUNT_FILE = Path(__import__('os').environ['GOOGLE_APPLICATION_CREDENTIALS'])

PROJECT_ID = __import__('os').environ['GOOGLE_CLOUD_PROJECT']

#Συνήθως δουλεύει το us-central1 για Vertex AI Gemini.
#Αν δεν δουλέψει, ρώτα τον καθηγητή ποιο location έχει ενεργοποιηθεί.
LOCATION = "us-central1"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

gemini_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    credentials=credentials
)

print("Vertex AI Gemini client initialized.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)

In [ ]:
#Gemini smoke test
#Τρέξε το μόνο αν άλλαξες model/key και θες να ελέγξεις αν δουλεύει το API.
#Κάθε smoke test καταναλώνει quota.

if RUN_SMOKE_TEST:

    print("Running Gemini smoke test with model:", AUG_MODEL)

    test_response = gemini_client.models.generate_content(
        model=AUG_MODEL,
        contents='Return only this JSON: {"ok": true}',
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=50,
            response_mime_type="application/json",
        ),
    )

    print(test_response.text)

else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True only when needed.")

In [ ]:
#helper functions

#Επιστρέφει ποιο κείμενο πρέπει να χρησιμοποιηθεί ως reference.
#Για PG το reference είναι το src, γιατί το task είναι paraphrase generation.
#Για MT/DM το reference είναι το tgt, γιατί εκεί βρίσκεται η επιθυμητή έξοδος.
def get_reference_text(row):

    if "ref" in row.index and pd.notna(row["ref"]):
        ref_field = str(row["ref"]).strip()

        if ref_field == "src":
            return "" if pd.isna(row["src"]) else str(row["src"])

        if ref_field == "tgt":
            return "" if pd.isna(row["tgt"]) else str(row["tgt"])

    if row["task"] == "PG":
        return "" if pd.isna(row["src"]) else str(row["src"])

    return "" if pd.isna(row["tgt"]) else str(row["tgt"])


#Καθαρίζει απλό text field.
#Χρησιμοποιείται πριν μπουν texts στα prompts ή στα CSVs.
def clean_text_value(x):

    if pd.isna(x):
        return ""

    return str(x).strip()


#Προσπαθεί να κάνει parse την απάντηση του LLM ως JSON.
#Υποστηρίζει και JSON object και JSON array, γιατί το batch generation επιστρέφει λίστα.
def safe_parse_json(text):

    if text is None:
        return None

    try:
        return json.loads(text)
    except Exception:
        pass

    #Πρώτα δοκιμάζουμε JSON object.
    try:
        start = text.find("{")
        end = text.rfind("}") + 1

        if start >= 0 and end > start:
            json_text = text[start:end]
            return json.loads(json_text)

    except Exception:
        pass

    #Μετά δοκιμάζουμε JSON array, χρήσιμο για batch generation.
    try:
        start = text.find("[")
        end = text.rfind("]") + 1

        if start >= 0 and end > start:
            json_text = text[start:end]
            return json.loads(json_text)

    except Exception:
        return None

    return None


#Fallback token length για περίπτωση που δεν υπάρχει ήδη token_len από προηγούμενα cells.
if "token_len" not in globals():

    def token_len(text):
        return len(str(text).split())


In [ ]:
#function to build augmentation prompt
#Single-sample prompt για συμβατότητα/μελλοντική χρήση.
#Στο κύριο pipeline χρησιμοποιείται batch prompt, αλλά η λογική faithful/hallucinated είναι ίδια.

def build_augmentation_prompt(row):

    task = clean_text_value(row["task"])
    src = clean_text_value(row["src"])
    tgt = clean_text_value(row["tgt"])
    reference_text = get_reference_text(row)

    #Task-specific οδηγίες ώστε το Gemini να ξέρει ποιο reference πρέπει να ακολουθήσει.
    if task == "PG":
        task_instruction = (
            "Task: Paraphrase Generation. "
            "Use the source as the reference. "
        )

    elif task == "MT":
        task_instruction = (
            "Task: Machine Translation. "
            "Use the target as the reference translation. "
        )

    elif task == "DM":
        task_instruction = (
            "Task: Definition Modeling. "
            "Use the target as the correct definition. "
        )

    else:
        task_instruction = (
            "Task: Natural Language Generation. "
            "Use the reference as the semantic ground truth. "
        )

    prompt = f"""
Generate synthetic data for hallucination detection.

{task_instruction}

Source:
{src}

Target:
{tgt}

Reference:
{reference_text}

Return two short English outputs:
- faithful: a paraphrase of the reference, not an exact copy. It must preserve the same meaning but use different wording where possible. If a paraphrase is difficult, make a minimal wording change without adding or removing meaning.
- hallucinated: fluent and plausible, but unsupported, incorrect, or meaning-changing. The hallucination should be subtle, not absurd.

Do not explain.
Return only valid JSON with keys: faithful, hallucinated.
"""

    return prompt


In [ ]:
#Gemini structured output schemas and generation functions
#Τα schemas βοηθούν το Gemini να επιστρέφει σταθερό JSON που γίνεται εύκολα parse.

AUG_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "faithful": {"type": "string"},
        "hallucinated": {"type": "string"}
    },
    "required": ["faithful", "hallucinated"]
}


BATCH_RESPONSE_SCHEMA = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "sample_id": {"type": "integer"},
            "faithful": {"type": "string"},
            "hallucinated": {"type": "string"}
        },
        "required": ["sample_id", "faithful", "hallucinated"]
    }
}


#helper function to extract text from Gemini response
#Χρησιμοποιείται γιατί μερικές φορές το response.text είναι κενό και το text βρίσκεται στα candidates/parts.
def extract_gemini_text(response):

    if getattr(response, "text", None) is not None:
        return response.text.strip()

    try:
        candidates = getattr(response, "candidates", None)

        if candidates and len(candidates) > 0:
            content = getattr(candidates[0], "content", None)
            parts = getattr(content, "parts", None)

            if parts:
                texts = []

                for part in parts:
                    part_text = getattr(part, "text", None)

                    if part_text is not None:
                        texts.append(part_text)

                if len(texts) > 0:
                    return "\n".join(texts).strip()

    except Exception:
        pass

    return None


#Κόβουμε πολύ μεγάλα texts ώστε το batch prompt να μένει μικρό και σταθερό.
def truncate_text(text, max_chars=1200):

    text = clean_text_value(text)

    if len(text) <= max_chars:
        return text

    return text[:max_chars] + "..."


#function to build batch augmentation prompt
#Εδώ φτιάχνουμε ένα prompt για πολλά samples μαζί, ώστε να μειώνονται τα API requests.
def build_batch_augmentation_prompt(batch_df):

    items = []

    for _, row in batch_df.iterrows():

        item = {
            "sample_id": int(row["sample_id"]),
            "task": clean_text_value(row["task"]),
            "src": truncate_text(row["src"]),
            "tgt": truncate_text(row["tgt"]),
            "reference": truncate_text(get_reference_text(row))
        }

        items.append(item)

    prompt = f"""
Generate synthetic training data for hallucination detection.

For each input item, create exactly two short English outputs:
1. faithful: a paraphrase of the reference, not an exact copy. It must preserve the same meaning but use different wording where possible. If a paraphrase is difficult, make a minimal wording change without adding or removing meaning.
2. hallucinated: fluent and plausible, but unsupported, incorrect, or meaning-changing. The hallucination should be subtle, not absurd.

Rules:
- Do not explain.
- Keep each output short.
- Return only valid JSON.
- Return one object per input item.
- Preserve the sample_id.

Input items:
{json.dumps(items, ensure_ascii=False, indent=2)}

Return JSON array with this schema:
[
  {{
    "sample_id": 0,
    "faithful": "...",
    "hallucinated": "..."
  }}
]
"""

    return prompt


#function to call Gemini for one synthetic pair with retries
#Το κρατάμε ως fallback/debugging, αλλά το κανονικό pipeline χρησιμοποιεί batch generation.
def generate_synthetic_pair(row, model=None, max_retries=None, retry_sleep=None):

    if model is None:
        model = AUG_MODEL

    if max_retries is None:
        max_retries = MAX_BATCH_RETRIES

    if retry_sleep is None:
        retry_sleep = RETRY_SLEEP_SECONDS

    prompt = build_augmentation_prompt(row)

    last_error = None

    for attempt in range(max_retries):

        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.4,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=AUG_RESPONSE_SCHEMA,
                ),
            )

            raw_text = extract_gemini_text(response)

            if raw_text is None or raw_text.strip() == "":
                last_error = "empty_response"
                print(f"Empty Gemini response for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            parsed = safe_parse_json(raw_text)

            if parsed is None:
                last_error = raw_text[:500]
                print(f"JSON parse failed for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            return parsed, raw_text

        except Exception as e:

            last_error = str(e)
            error_text = str(e)

            if (
                "401" in error_text
                or "403" in error_text
                or "UNAUTHENTICATED" in error_text
                or "PERMISSION_DENIED" in error_text
                or "API key not valid" in error_text
            ):
                raise ValueError(
                    "Gemini credentials are invalid or do not have permission."
                ) from e

            print(f"Gemini single-sample error for {model} | attempt {attempt + 1}/{max_retries}")
            print(error_text[:300])
            time.sleep(retry_sleep)

    return None, last_error


#function to call Gemini for a batch of samples
#Fail-fast: λίγα retries και μικρή αναμονή. Αν αποτύχει, το batch γράφεται στο failed log.
def generate_synthetic_batch(batch_df, model=None, max_retries=None, retry_sleep=None):

    if model is None:
        model = AUG_MODEL

    if max_retries is None:
        max_retries = MAX_BATCH_RETRIES

    if retry_sleep is None:
        retry_sleep = RETRY_SLEEP_SECONDS

    prompt = build_batch_augmentation_prompt(batch_df)
    last_error = None

    for attempt in range(max_retries):

        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.4,
                    max_output_tokens=5000,
                    response_mime_type="application/json",
                    response_schema=BATCH_RESPONSE_SCHEMA,
                ),
            )

            raw_text = extract_gemini_text(response)

            if raw_text is None or raw_text.strip() == "":
                last_error = "empty_response"
                print(f"Empty Gemini batch response for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            parsed = safe_parse_json(raw_text)

            if parsed is None:
                last_error = raw_text[:500]
                print(f"Batch JSON parse failed for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            if not isinstance(parsed, list):
                last_error = str(parsed)[:500]
                print(f"Batch response is not a list for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            return parsed, raw_text

        except Exception as e:

            last_error = str(e)
            error_text = str(e)

            if (
                "401" in error_text
                or "403" in error_text
                or "UNAUTHENTICATED" in error_text
                or "PERMISSION_DENIED" in error_text
                or "API key not valid" in error_text
            ):
                raise ValueError(
                    "Gemini credentials are invalid or do not have permission."
                ) from e

            if (
                "429" in error_text
                or "503" in error_text
                or "UNAVAILABLE" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
                or "quota" in error_text.lower()
                or "high demand" in error_text.lower()
            ):
                print(f"Gemini temporary limit/high demand for {model} | attempt {attempt + 1}/{max_retries}")
                print(f"Waiting {retry_sleep} seconds...")
                time.sleep(retry_sleep)
                continue

            print(f"Gemini batch error for {model} | attempt {attempt + 1}/{max_retries}")
            print(error_text[:300])
            time.sleep(retry_sleep)

    return None, last_error


In [ ]:
#Choosing dataset for augmentation
#Εδώ επιλέγουμε μικρό balanced subset από το unlabeled train set.
#Το subset είναι balanced ανά task ώστε το augmentation να καλύπτει DM, MT και PG.

#Χρησιμοποιούμε μόνο unlabeled train data.
#Δεν χρησιμοποιούμε validation/test data για synthetic training data.
augmentation_base_df = train_agnostic_df.copy()

#Έλεγχος ότι υπάρχουν οι απαραίτητες στήλες.
required_cols = ["task", "src", "hyp", "tgt"]
missing_cols = [col for col in required_cols if col not in augmentation_base_df.columns]

if len(missing_cols) > 0:
    raise ValueError(f"Missing columns for augmentation: {missing_cols}")

#Κρατάμε stable sample id από το αρχικό index.
#Έτσι αν αλλάξεις SAMPLES_PER_TASK από 5 σε 20 ή 100 δεν μπερδεύεται το checkpoint.
augmentation_base_df = augmentation_base_df.reset_index().rename(columns={"index": "sample_id"})

#Καθαρίζουμε τα βασικά text fields.
for col in ["task", "src", "hyp", "tgt"]:
    augmentation_base_df[col] = augmentation_base_df[col].fillna("").astype(str).str.strip()

augmentation_base_df["sample_id"] = augmentation_base_df["sample_id"].astype(int)

#Κρατάμε μόνο τα τρία SHROOM tasks.
augmentation_base_df = augmentation_base_df[
    augmentation_base_df["task"].isin(["DM", "MT", "PG"])
].copy()

if augmentation_base_df.empty:
    raise ValueError("No rows left after task filtering.")

#Αν δεν υπάρχει ref, το δημιουργούμε με βάση το task.
if "ref" not in augmentation_base_df.columns:
    augmentation_base_df["ref"] = np.where(
        augmentation_base_df["task"] == "PG",
        "src",
        "tgt"
    )

#Κρατάμε balanced δείγμα ανά task με σταθερό random_state για αναπαραγωγιμότητα.
sample_parts = []

for task_name, task_df in augmentation_base_df.groupby("task"):

    n_samples = min(SAMPLES_PER_TASK, len(task_df))

    sampled_task_df = task_df.sample(
        n=n_samples,
        random_state=42
    )

    sample_parts.append(sampled_task_df)

sample_unlabeled = pd.concat(
    sample_parts,
    ignore_index=True
)

print("Sample for augmentation shape:", sample_unlabeled.shape)
print(sample_unlabeled["task"].value_counts().to_string())

display(sample_unlabeled.head())


In [ ]:
#Synthetic data generation with batch checkpoint/resume
#Κύριο generation loop: διαβάζει checkpoint, τρέχει μόνο τα remaining samples και σώζει πρόοδο μετά από κάθε batch.

raw_save_path = AUG_OUTPUT_DIR / "synthetic_raw_data.csv"
failed_save_path = AUG_OUTPUT_DIR / "synthetic_failed_generation.csv"
CHECKPOINT_MODE = "gemini_batch_v3"

#Recover προηγούμενο checkpoint, αν υπάρχει.
#Έτσι μπορούμε να σταματήσουμε/ξανατρέξουμε χωρίς να χαθεί η πρόοδος.
if raw_save_path.exists() and raw_save_path.stat().st_size > 0:

    try:
        synthetic_raw_df = pd.read_csv(raw_save_path)

        #Αν το checkpoint είναι από παλιό generation mode, το αγνοούμε για να μη μπερδευτούν sample_ids.
        if "generation_mode" in synthetic_raw_df.columns:
            synthetic_raw_df = synthetic_raw_df[
                synthetic_raw_df["generation_mode"] == CHECKPOINT_MODE
            ].copy()
        else:
            synthetic_raw_df = pd.DataFrame()

        if not synthetic_raw_df.empty and "sample_id" in synthetic_raw_df.columns:
            completed_sample_ids = set(
                synthetic_raw_df["sample_id"].dropna().astype(int).tolist()
            )
            synthetic_records = synthetic_raw_df.to_dict("records")
        else:
            completed_sample_ids = set()
            synthetic_records = []

    except pd.errors.EmptyDataError:
        synthetic_records = []
        completed_sample_ids = set()

else:
    synthetic_records = []
    completed_sample_ids = set()

failed_records = []

remaining_df = sample_unlabeled[
    ~sample_unlabeled["sample_id"].isin(completed_sample_ids)
].copy()

print("\n----------------------------------")
print("Starting/resuming batch synthetic data generation...")
print("Already completed:", len(completed_sample_ids))
print("Remaining samples:", len(remaining_df))
print("Batch size:", BATCH_SIZE)
print("Selected model:", AUG_MODEL)
print("----------------------------------")

start_time = time.time()

batch_starts = list(range(0, len(remaining_df), BATCH_SIZE))

for start_idx in tqdm(batch_starts, desc="Generating synthetic batches"):

    batch_df = remaining_df.iloc[start_idx:start_idx + BATCH_SIZE].copy()

    #Κρατάμε μόνο samples που έχουν source και reference.
    valid_rows = []

    for _, row in batch_df.iterrows():

        sample_id = int(row["sample_id"])
        src = clean_text_value(row["src"])
        reference_text = get_reference_text(row)

        if src == "" or reference_text == "":
            failed_records.append({
                "sample_id": sample_id,
                "task": clean_text_value(row["task"]),
                "reason": "empty_source_or_reference",
                "error_type": "data",
                "raw_response": "",
                "timestamp": datetime.datetime.now().isoformat()
            })
        else:
            valid_rows.append(row)

    if len(valid_rows) == 0:
        continue

    batch_df = pd.DataFrame(valid_rows)

    parsed_batch, response_or_error = generate_synthetic_batch(
        batch_df,
        model=AUG_MODEL,
        max_retries=MAX_BATCH_RETRIES,
        retry_sleep=RETRY_SLEEP_SECONDS
    )

    #Αν αποτύχει το batch, δοκιμάζουμε single-sample fallback.
    if parsed_batch is None:

        print("Batch failed. Trying single-sample fallback...")

        recovered_records = []

        for _, row in batch_df.iterrows():

            sample_id = int(row["sample_id"])

            parsed_single, raw_single = generate_synthetic_pair(
                row,
                model=AUG_MODEL,
                max_retries=1,
                retry_sleep=RETRY_SLEEP_SECONDS
            )

            if parsed_single is None:
                failed_records.append({
                    "sample_id": sample_id,
                    "task": clean_text_value(row["task"]),
                    "reason": "single_generation_failed_after_batch_failed",
                    "error_type": "generation",
                    "raw_response": response_or_error,
                    "timestamp": datetime.datetime.now().isoformat()
                })
                continue

            recovered_records.append((row, parsed_single))

        parsed_batch = []

        for row, parsed_single in recovered_records:
            parsed_batch.append({
                "sample_id": int(row["sample_id"]),
                "faithful": parsed_single.get("faithful", ""),
                "hallucinated": parsed_single.get("hallucinated", "")
            })

        if len(parsed_batch) == 0:
            pd.DataFrame(synthetic_records).to_csv(raw_save_path, index=False)

            if len(failed_records) > 0:
                pd.DataFrame(failed_records).to_csv(failed_save_path, index=False)

            time.sleep(REQUEST_SLEEP)
            continue

    parsed_by_id = {}

    for item in parsed_batch:

        try:
            parsed_sample_id = int(item.get("sample_id"))
            parsed_by_id[parsed_sample_id] = item
        except Exception:
            continue

    for _, row in batch_df.iterrows():

        sample_id = int(row["sample_id"])

        if sample_id not in parsed_by_id:
            failed_records.append({
                "sample_id": sample_id,
                "task": clean_text_value(row["task"]),
                "reason": "missing_sample_in_batch_response",
                "error_type": "parsing",
                "raw_response": "",
                "timestamp": datetime.datetime.now().isoformat()
            })
            continue

        item = parsed_by_id[sample_id]

        faithful = clean_text_value(item.get("faithful", ""))
        hallucinated = clean_text_value(item.get("hallucinated", ""))

        reference_text = get_reference_text(row)

        base_record = {
            "sample_id": sample_id,
            "task": clean_text_value(row["task"]),
            "src": clean_text_value(row["src"]),
            "tgt": clean_text_value(row["tgt"]),
            "reference_text": reference_text,
            "original_hyp": clean_text_value(row["hyp"]) if "hyp" in row.index else "",
            "synthetic": True,
            "generation_mode": CHECKPOINT_MODE
        }

        #Φάση Α: faithful output -> Not Hallucination
        if faithful != "":
            synthetic_records.append({
                **base_record,
                "hyp": faithful,
                "label": "Not Hallucination",
                "label_bin": 0,
                "p_hallucination": 0.0,
                "source_type": "synthetic_faithful"
            })

        #Φάση Β: hallucinated output -> Hallucination
        if hallucinated != "":
            synthetic_records.append({
                **base_record,
                "hyp": hallucinated,
                "label": "Hallucination",
                "label_bin": 1,
                "p_hallucination": 1.0,
                "source_type": "synthetic_hallucination"
            })

    #Checkpoint μετά από κάθε batch.
    pd.DataFrame(synthetic_records).to_csv(raw_save_path, index=False)

    if len(failed_records) > 0:
        pd.DataFrame(failed_records).to_csv(failed_save_path, index=False)

    time.sleep(REQUEST_SLEEP)

end_time = time.time()

synthetic_raw_df = pd.DataFrame(synthetic_records)
synthetic_df = synthetic_raw_df.copy()

print("\n----------------------------------")
print(f"Χρόνος παραγωγής: {(end_time - start_time) / 60:.2f} λεπτά")
print("Generated synthetic shape:", synthetic_raw_df.shape)
print("Failed generations:", len(failed_records))
print("----------------------------------")

if not synthetic_raw_df.empty:
    display(synthetic_raw_df.head())
else:
    print("No synthetic samples were generated.")


In [ ]:
#Load semantic similarity model if needed

if "sim_model" not in globals():
    from sentence_transformers import SentenceTransformer

    SIM_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
    sim_model = SentenceTransformer(SIM_MODEL_NAME)

    print("Semantic similarity model loaded:", SIM_MODEL_NAME)
else:
    print("Using existing semantic similarity model.")


In [ ]:
### Strict Filtering Phase

print("\n----------------------------------")
print("Filtering synthetic data...")
print("----------------------------------")

#Ξεκινάμε από το raw synthetic data που παρήχθη στο προηγούμενο βήμα.
synthetic_df = synthetic_raw_df.copy()
print("Before filtering:", synthetic_df.shape)

#Σταθερή δομή για τα τελικά synthetic data.
final_columns = [
    "sample_id", "task", "src", "tgt", "hyp", "label", "label_bin",
    "p_hallucination", "source_type", "synthetic", "reference_text",
    "original_hyp", "generation_mode", "sim_ref_hyp"
]

#Αν δεν υπάρχουν synthetic records, δημιουργούμε κενά DataFrames ώστε να μη σπάσουν τα επόμενα cells.
if synthetic_df.empty:
    print("\nNo synthetic data generated. Creating empty final_synthetic_df.")
    final_synthetic_df = pd.DataFrame(columns=final_columns)
    final_synthetic_df_balanced = final_synthetic_df.copy()

else:
    #Βεβαιωνόμαστε ότι υπάρχει reference_text.
    #Για PG reference = src, ενώ για DM/MT reference = tgt.
    if "reference_text" not in synthetic_df.columns:
        synthetic_df["reference_text"] = synthetic_df.apply(get_reference_text, axis=1)

    #Καθαρίζουμε βασικά πεδία πριν από τα filters.
    for col in ["src", "tgt", "hyp", "reference_text"]:
        for col in ["src", "tgt", "hyp", "reference_text"]:
            synthetic_df[col] = synthetic_df[col].fillna("").astype(str).str.strip()

    #Αφαιρούμε άδεια hypotheses.
    synthetic_df = synthetic_df[
        synthetic_df["hyp"] != ""
    ].copy()

    #Υπολογισμός απλού token length για basic sanity checks.
    synthetic_df["hyp_tokens"] = synthetic_df["hyp"].apply(token_len)
    synthetic_df["reference_tokens"] = synthetic_df["reference_text"].apply(token_len)

    #Basic filtering βάσει μήκους.
    #Πετάμε outputs που είναι υπερβολικά μικρά ή υπερβολικά μεγάλα.
    synthetic_df = synthetic_df[
        (synthetic_df["hyp_tokens"] >= 3)
        & (synthetic_df["hyp_tokens"] <= 200)
    ].copy()

    #Exact-copy diagnostics πριν αφαιρέσουμε τα αντίγραφα.
    #Αυτό δείχνει αν το faithful generation αντιγράφει απλώς το reference.
    exact_copy_mask = (
        synthetic_df["hyp"].str.lower().str.strip()
        == synthetic_df["reference_text"].str.lower().str.strip()
    )

    print("\nExact-copy diagnostics:")
    print("Total exact copies:", int(exact_copy_mask.sum()))

    if "source_type" in synthetic_df.columns:
        print("Exact copies by source_type:")
        print(synthetic_df[exact_copy_mask]["source_type"].value_counts(dropna=False).to_string())

    #Αφαιρούμε outputs που είναι ακριβώς ίδια με το reference.
    #Δεν προσφέρουν πραγματικό augmentation.
    synthetic_df = synthetic_df[~exact_copy_mask].copy()

    #Αφαιρούμε outputs που είναι υπερβολικά μεγαλύτερα από το reference.
    synthetic_df = synthetic_df[
        synthetic_df["hyp_tokens"] <= 3 * synthetic_df["reference_tokens"].clip(lower=1)
    ].copy()

    print("After basic filtering:", synthetic_df.shape)

    if synthetic_df.empty:
        print("\nNo data after basic filtering. Creating empty final_synthetic_df.")
        final_synthetic_df = pd.DataFrame(columns=final_columns)
        final_synthetic_df_balanced = final_synthetic_df.copy()

    else:
        #Semantic similarity filtering.
        #Χρησιμοποιούμε reference_text και όχι πάντα src, ώστε το filtering να είναι σωστό για όλα τα tasks.
        syn_ref_emb = sim_model.encode(
            synthetic_df["reference_text"].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        syn_hyp_emb = sim_model.encode(
            synthetic_df["hyp"].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        #Cosine similarity, επειδή τα embeddings είναι normalized.
        synthetic_df["sim_ref_hyp"] = np.sum(
            syn_ref_emb * syn_hyp_emb,
            axis=1
        )

        #Thresholds σύμφωνα με τη λογική του οδηγού.
        #Hallucination: αρκετά κοντά ώστε να είναι ρεαλιστικό, όχι τόσο κοντά ώστε να είναι faithful.
        #Not Hallucination: υψηλή similarity με το reference.
        HALLUCINATION_MIN_SIM = 0.30
        HALLUCINATION_MAX_SIM = 0.95

        FAITHFUL_MIN_SIM = 0.75
        FAITHFUL_MAX_SIM = 0.99

        hallucination_mask = (
            (synthetic_df["label_bin"] == 1)
            & (synthetic_df["sim_ref_hyp"] >= HALLUCINATION_MIN_SIM)
            & (synthetic_df["sim_ref_hyp"] <= HALLUCINATION_MAX_SIM)
        )

        faithful_mask = (
            (synthetic_df["label_bin"] == 0)
            & (synthetic_df["sim_ref_hyp"] >= FAITHFUL_MIN_SIM)
            & (synthetic_df["sim_ref_hyp"] <= FAITHFUL_MAX_SIM)
        )

        final_synthetic_df = synthetic_df[
            hallucination_mask | faithful_mask
        ].copy()

        print("After semantic filtering:", final_synthetic_df.shape)

        print("\nLabel distribution:")
        print(final_synthetic_df["label"].value_counts(dropna=False).to_string())

        print("\nSimilarity statistics by source_type:")
        if not final_synthetic_df.empty and "source_type" in final_synthetic_df.columns:
            print(
                final_synthetic_df
                .groupby("source_type")["sim_ref_hyp"]
                .describe()
                .round(3)
                .to_string()
            )

        #Δημιουργούμε balanced version για training.
        #Το filtered dataset μπορεί να είναι ανισόρροπο, επειδή το filtering πετάει περισσότερα faithful samples.
        label_counts = final_synthetic_df["label"].value_counts(dropna=False)

        if len(label_counts) == 2:
            min_count = label_counts.min()

            final_synthetic_df_balanced = (
                final_synthetic_df
                .groupby("label", group_keys=False)
                .apply(lambda x: x.sample(n=min_count, random_state=42))
                .reset_index(drop=True)
            )
        else:
            print("\nWarning: only one label exists after filtering. Balanced dataset will be identical to filtered dataset.")
            final_synthetic_df_balanced = final_synthetic_df.copy()

print("\n----------------------------------")
print("Filtered synthetic shape:", final_synthetic_df.shape)
print("Balanced synthetic shape:", final_synthetic_df_balanced.shape)

if not final_synthetic_df.empty:
    display(final_synthetic_df.head())
else:
    print("Final synthetic DataFrame is empty. Skipping display.")
print("----------------------------------")


In [ ]:
#Manual inspection sample

#Παίρνουμε λίγα δείγματα για χειροκίνητο έλεγχο ποιότητας.
#Αν το filtered synthetic dataset είναι άδειο, δεν κάνουμε inspection.

if "final_synthetic_df" not in globals():
    print("final_synthetic_df does not exist. Run the filtering cell first.")

elif final_synthetic_df.empty:
    print("No synthetic data to inspect (final_synthetic_df is empty).")
    print("Check synthetic_raw_data.csv, failed generations, and filtering thresholds before continuing.")

else:
    inspection_cols = [
        "task",
        "label",
        "source_type",
        "sim_ref_hyp",
        "reference_text",
        "hyp"
    ]

    available_cols = [
        col for col in inspection_cols
        if col in final_synthetic_df.columns
    ]

    inspection_parts = []

    for _, group_df in final_synthetic_df.groupby(["task", "label"]):

        n_samples = min(3, len(group_df))

        if n_samples > 0:
            inspection_parts.append(
                group_df.sample(n=n_samples, random_state=42)
            )

    if len(inspection_parts) == 0:
        print("No synthetic data to inspect after grouping.")

    else:
        inspection_df = pd.concat(
            inspection_parts,
            ignore_index=True
        )

        if len(available_cols) == 0:
            print("No inspection columns are available.")
            print("Available columns:")
            print(final_synthetic_df.columns.tolist())

        else:
            display(inspection_df[available_cols])


In [ ]:
#Saving the final synthetic dataset

#Σώζουμε και το filtered dataset και το balanced dataset.
#Για training χρησιμοποιούμε συνήθως το balanced version.

if "final_synthetic_df" not in globals():
    raise ValueError("final_synthetic_df does not exist. Run the filtering cell first.")

expected_synthetic_cols = [
    "sample_id",
    "task",
    "src",
    "tgt",
    "reference_text",
    "original_hyp",
    "hyp",
    "label",
    "label_bin",
    "p_hallucination",
    "source_type",
    "synthetic",
    "generation_mode",
    "sim_ref_hyp"
]

#Αν το filtered dataframe είναι άδειο, κρατάμε τις αναμενόμενες στήλες.
if final_synthetic_df.empty:
    print("final_synthetic_df is empty. Saving empty filtered/balanced files.")
    final_synthetic_df = pd.DataFrame(columns=expected_synthetic_cols)

#Αν λείπουν στήλες από το filtered dataframe, τις προσθέτουμε.
for col in expected_synthetic_cols:
    if col not in final_synthetic_df.columns:
        final_synthetic_df[col] = np.nan

#Δημιουργούμε ξανά balanced dataframe με ασφαλή τρόπο.
if not final_synthetic_df.empty and "label" in final_synthetic_df.columns:

    label_counts = final_synthetic_df["label"].value_counts(dropna=False)

    if len(label_counts) == 2:
        min_count = label_counts.min()

        final_synthetic_df_balanced = (
            final_synthetic_df
            .groupby("label", group_keys=False)
            .apply(lambda x: x.sample(n=min_count, random_state=42))
            .reset_index(drop=True)
        )

    else:
        print("Warning: only one label exists after filtering.")
        print("Balanced dataset will be identical to filtered dataset.")
        final_synthetic_df_balanced = final_synthetic_df.copy()

else:
    print("Warning: no valid filtered synthetic samples.")
    final_synthetic_df_balanced = pd.DataFrame(columns=expected_synthetic_cols)

#Βεβαιωνόμαστε ότι και το balanced dataframe έχει τις αναμενόμενες στήλες.
for col in expected_synthetic_cols:
    if col not in final_synthetic_df_balanced.columns:
        final_synthetic_df_balanced[col] = np.nan

final_synthetic_path = AUG_OUTPUT_DIR / "filtered_synthetic_data.csv"
final_synthetic_jsonl_path = AUG_OUTPUT_DIR / "filtered_synthetic_data.jsonl"

final_synthetic_balanced_path = AUG_OUTPUT_DIR / "filtered_synthetic_data_balanced.csv"
final_synthetic_balanced_jsonl_path = AUG_OUTPUT_DIR / "filtered_synthetic_data_balanced.jsonl"

#Βεβαιωνόμαστε ότι το label υπάρχει σωστά και στο balanced dataset.
if "label" not in final_synthetic_df_balanced.columns or final_synthetic_df_balanced["label"].isna().any():
    final_synthetic_df_balanced["label"] = final_synthetic_df_balanced["label_bin"].map({
        1: "Hallucination",
        0: "Not Hallucination"
    })

#Βεβαιωνόμαστε ότι το label υπάρχει σωστά και στο filtered dataset.
if "label" not in final_synthetic_df.columns or final_synthetic_df["label"].isna().any():
    final_synthetic_df["label"] = final_synthetic_df["label_bin"].map({
        1: "Hallucination",
        0: "Not Hallucination"
    })
    
final_synthetic_df.to_csv(final_synthetic_path, index=False)
final_synthetic_df.to_json(
    final_synthetic_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

final_synthetic_df_balanced.to_csv(final_synthetic_balanced_path, index=False)
final_synthetic_df_balanced.to_json(
    final_synthetic_balanced_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("\n" + "="*60)
print("FINAL SYNTHETIC DATA SUMMARY")
print("="*60)

print("\nFiltered Dataset (final_synthetic_df):")
print(f"  Shape: {final_synthetic_df.shape}")

if not final_synthetic_df.empty and "label" in final_synthetic_df.columns:
    print("  Label distribution:")
    print(final_synthetic_df["label"].value_counts(dropna=False).to_string())
else:
    print("  No labels available or dataset is empty.")

print("\nBalanced Dataset (final_synthetic_df_balanced):")
print(f"  Shape: {final_synthetic_df_balanced.shape}")

if not final_synthetic_df_balanced.empty and "label" in final_synthetic_df_balanced.columns:
    print("  Label distribution:")
    print(final_synthetic_df_balanced["label"].value_counts(dropna=False).to_string())
else:
    print("  No labels available or dataset is empty.")

print("\n" + "-"*60)
print("Saved Files:")
print("-"*60)
print(f"Filtered CSV:        {final_synthetic_path.resolve()}")
print(f"Filtered JSONL:      {final_synthetic_jsonl_path.resolve()}")
print(f"Balanced CSV:        {final_synthetic_balanced_path.resolve()}")
print(f"Balanced JSONL:      {final_synthetic_balanced_jsonl_path.resolve()}")
print("="*60)

if not final_synthetic_df.empty:
    display(final_synthetic_df.head())
else:
    print("(Final synthetic DataFrame is empty)")

In [ ]:
#check: Θα μας πει αν το filtering κράτησε καλά δείγματα.
filtered_df = pd.read_csv(AUG_OUTPUT_DIR / "filtered_synthetic_data.csv")
balanced_df = pd.read_csv(AUG_OUTPUT_DIR / "filtered_synthetic_data_balanced.csv")

print("Filtered shape:", filtered_df.shape)
print(filtered_df["label"].value_counts(dropna=False) if "label" in filtered_df.columns else "No label column")

print("\nBalanced shape:", balanced_df.shape)
print(balanced_df["label"].value_counts(dropna=False) if "label" in balanced_df.columns else "No label column")

In [ ]:
#Optional: combine synthetic data with a human-labeled dataframe for later training

#Χρησιμοποιούμε το balanced synthetic dataset για training by default,
#ώστε να μην εισάγουμε class imbalance.
#Αν δεν υπάρχουν synthetic samples ή labeled dataframe, το combine γίνεται skip.

required_synthetic_cols = [
    "task",
    "src",
    "tgt",
    "hyp",
    "label",
    "label_bin",
    "p_hallucination",
    "source_type",
    "synthetic",
    "reference_text"
]

#Επιλέγουμε synthetic dataset.
if (
    "final_synthetic_df_balanced" in globals()
    and isinstance(final_synthetic_df_balanced, pd.DataFrame)
    and not final_synthetic_df_balanced.empty
    and all(col in final_synthetic_df_balanced.columns for col in required_synthetic_cols)
):
    synthetic_source_df = final_synthetic_df_balanced.copy()
    print("Using final_synthetic_df_balanced for training.")

elif (
    "final_synthetic_df" in globals()
    and isinstance(final_synthetic_df, pd.DataFrame)
    and not final_synthetic_df.empty
    and all(col in final_synthetic_df.columns for col in required_synthetic_cols)
):
    synthetic_source_df = final_synthetic_df.copy()
    print("Using final_synthetic_df for training because balanced dataset is not available.")

else:
    synthetic_source_df = pd.DataFrame()
    print("No valid synthetic dataset available for combine.")
    print("Skipping augmented_training_df creation.")
    print("Check generation/filtering first.")


#Επιλέγουμε human-labeled dataframe.
#Προσοχή: δεν χρησιμοποιούμε αυτόματα official test data για training.
#Για ασφαλές demo χρησιμοποιούμε trial_df αν υπάρχει.
if "human_labeled_df" in globals():
    base_human_df = human_labeled_df.copy()
    print("Using existing human_labeled_df.")

elif "trial_df" in globals():
    base_human_df = trial_df.copy()
    print("Using trial_df as human-labeled dataframe.")

else:
    base_human_df = pd.DataFrame()
    print("No human-labeled dataframe available.")


if synthetic_source_df.empty:
    print("Combine skipped because synthetic_source_df is empty.")

elif base_human_df.empty:
    print("Combine skipped because human-labeled dataframe is empty.")

else:
    #Κανονικοποιούμε το probability column.
    if "p_hallucination" not in base_human_df.columns:
        if "p(Hallucination)" in base_human_df.columns:
            base_human_df["p_hallucination"] = base_human_df["p(Hallucination)"]
        else:
            raise ValueError("Missing p_hallucination or p(Hallucination) column.")

    #Κανονικοποιούμε το binary label.
    if "label_bin" not in base_human_df.columns:
        if "label" in base_human_df.columns:
            base_human_df["label_bin"] = base_human_df["label"].map({
                "Hallucination": 1,
                "Not Hallucination": 0
            })
        else:
            raise ValueError("Missing label column.")

    required_labeled_cols = [
        "task",
        "src",
        "tgt",
        "hyp",
        "label",
        "label_bin",
        "p_hallucination"
    ]

    missing_labeled_cols = [
        col for col in required_labeled_cols
        if col not in base_human_df.columns
    ]

    if len(missing_labeled_cols) > 0:
        raise ValueError(f"Missing columns from human-labeled dataframe: {missing_labeled_cols}")

    labeled_for_training_df = base_human_df[required_labeled_cols].copy()

    labeled_for_training_df["source_type"] = "human_labeled"
    labeled_for_training_df["synthetic"] = False
    labeled_for_training_df["reference_text"] = labeled_for_training_df.apply(get_reference_text, axis=1)

    synthetic_for_training_df = synthetic_source_df[required_synthetic_cols].copy()

    augmented_training_df = pd.concat(
        [labeled_for_training_df, synthetic_for_training_df],
        ignore_index=True
    )

    augmented_training_path = AUG_OUTPUT_DIR / "augmented_training_data.csv"

    augmented_training_df.to_csv(
        augmented_training_path,
        index=False
    )

    print("\nHuman labeled shape:", labeled_for_training_df.shape)
    print("Synthetic shape:", synthetic_for_training_df.shape)
    print("Augmented training shape:", augmented_training_df.shape)

    print("\nSource type distribution:")
    print(augmented_training_df["source_type"].value_counts(dropna=False).to_string())

    print("\nSaved:")
    print(augmented_training_path.resolve())

    display(augmented_training_df.head())


In [ ]:
#Debug

#search for generated files in project folders
root = Path((__import__('os').environ.get('SHROOM_PROJECT_ROOT', '.') + '/data'))

for p in root.rglob("synthetic_raw_data.csv"):
    print(p.resolve())
    
for p in root.rglob("filtered_synthetic_data.csv"):
    print(p.resolve())

In [ ]:
raw_df = pd.read_csv(AUG_OUTPUT_DIR / "synthetic_raw_data.csv")

print("Raw shape:", raw_df.shape)

print("\nRaw label distribution:")
print(raw_df["label"].value_counts(dropna=False).to_string())

print("\nRaw source_type distribution:")
print(raw_df["source_type"].value_counts(dropna=False).to_string())

exact_copy_mask = (
    raw_df["hyp"].str.lower().str.strip()
    == raw_df["reference_text"].str.lower().str.strip()
)

print("\nExact copies total:", exact_copy_mask.sum())

print("\nExact copies by source_type:")
print(raw_df[exact_copy_mask]["source_type"].value_counts(dropna=False).to_string())

display(raw_df.head())

In [ ]:
filtered_df = pd.read_csv(AUG_OUTPUT_DIR / "filtered_synthetic_data.csv")
balanced_df = pd.read_csv(AUG_OUTPUT_DIR / "filtered_synthetic_data_balanced.csv")

print("Filtered shape:", filtered_df.shape)
print(filtered_df["label"].value_counts(dropna=False) if "label" in filtered_df.columns else "No label column")

print("\nBalanced shape:", balanced_df.shape)
print(balanced_df["label"].value_counts(dropna=False) if "label" in balanced_df.columns else "No label column")